# IAN 601 · Module 11 — Advanced Pandas

**Reshape, Time, and Workflows**

This notebook follows the Module 11 lesson and works from a CSV dataset stored in the course GitHub repository.

[Open the Module 11 slides](https://ali2449.github.io/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/s_module11.html)

[View the CSV dataset on GitHub](https://github.com/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/blob/main/IAN601_Module11_Advanced_Pandas_Sales.csv)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/blob/main/IAN_601_Module_11_Advanced_Pandas.ipynb)

### Topics
- Read a CSV directly from GitHub
- Method chaining with `query()` and `assign()`
- `groupby()` with named aggregation
- `agg()` versus `transform()`
- Pivot tables, `melt()`, and `explode()`
- Datetime features, `resample()`, and rolling windows
- `map()`, `replace()`, and `cut()`
- Safer joins with `merge()`
- Reusable workflows with `pipe()`

## 1. Load the CSV dataset

The dataset is stored in the same GitHub repository as this notebook. Using the raw GitHub URL means the notebook works in Google Colab without asking students to upload the CSV manually.

The file contains 300 sales transactions with dates, customers, regions, channels, products, prices, discounts, sales amounts, and simple tags.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)

csv_url = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/"
    "main/IAN601_Module11_Advanced_Pandas_Sales.csv"
)

orders = pd.read_csv(csv_url)

print("Dataset shape:", orders.shape)
orders.head()

## 2. Method chaining

Method chaining keeps several related steps together. Read the code from top to bottom: start with the table, filter rows, create a column, and sort the result.

In [ ]:
result = (
    orders
    .query("amount >= 70")
    .assign(fee=lambda df: df["amount"] * 0.05)
    .sort_values("amount", ascending=False)
)

result[["order_id", "amount", "fee"]]

### Try it
Change the filter from `amount >= 70` to another value. Then change the fee from 5% to 8%.

## 3. `query()` and `assign()`

`query()` can make a filter easy to read. `assign()` adds a new column and allows the chain to continue.

In [ ]:
web_orders = (
    orders
    .query("channel == 'Web' and amount >= 50")
    .assign(tax=lambda df: df["amount"] * 0.07)
)

web_orders[["order_id", "channel", "amount", "tax"]]

## 4. Named aggregation with `groupby()`

A single `groupby()` can calculate several useful measures at the same time. Named aggregation gives the output columns clear names.

In [ ]:
region_summary = (
    orders
    .groupby("region", as_index=False)
    .agg(
        total_sales=("amount", "sum"),
        average_order=("amount", "mean"),
        orders=("order_id", "count")
    )
)

region_summary.round(2)

## 5. `agg()` versus `transform()`

Use `agg()` when you want a smaller summary table. Use `transform()` when you want a group statistic returned for every original row.

In [ ]:
orders_compare = orders.copy()

orders_compare["region_average"] = (
    orders_compare
    .groupby("region")["amount"]
    .transform("mean")
)

orders_compare["difference"] = (
    orders_compare["amount"] - orders_compare["region_average"]
)

orders_compare[
    ["order_id", "region", "amount", "region_average", "difference"]
].round(2)

Notice that `transform()` did **not** reduce the number of rows. Each order still appears once.

## 6. Pivot tables

A pivot table creates a matrix-like summary. Here, regions become rows and channels become columns.

In [ ]:
report = pd.pivot_table(
    orders,
    index="region",
    columns="channel",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

report

## 7. Reshape wide data with `melt()`

We can build a monthly sales summary from the CSV, pivot it into a wide spreadsheet-style table, and then use `melt()` to return it to long format.

In [ ]:
monthly_sales = (
    orders
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .assign(month=lambda df: df["date"].dt.strftime("%b"))
    .groupby(["region", "month"], as_index=False)["amount"]
    .sum()
)

wide = monthly_sales.pivot(
    index="region",
    columns="month",
    values="amount"
).fillna(0).reset_index()

wide

In [ ]:
long_sales = wide.melt(
    id_vars="region",
    var_name="month",
    value_name="sales"
)

long_sales.head(12)

## 8. One item per row with `explode()`

The CSV contains a `tags` column such as `student|learning`. First split the text into a Python list, then use `explode()` to create one row per tag.

In [ ]:
orders_tags = orders[["order_id", "product", "tags"]].copy()

orders_tags["tags"] = orders_tags["tags"].str.split("|")

orders_tags.head()

In [ ]:
tags_long = orders_tags.explode("tags")

tags_long.head(10)

In [ ]:
tags_long["tags"].value_counts()

## 9. Work with dates

Dates are often read as strings. Convert them to real datetime values before using time-based operations.

In [ ]:
orders_dates = orders.copy()
orders_dates["date"] = pd.to_datetime(orders_dates["date"])

orders_dates["weekday"] = orders_dates["date"].dt.day_name()
orders_dates["month"] = orders_dates["date"].dt.to_period("M")
orders_dates["day"] = orders_dates["date"].dt.day

orders_dates[["order_id", "date", "weekday", "month", "day"]]

## 10. Time-based summaries with `resample()`

`resample()` groups data into regular time periods. This example calculates daily sales totals.

In [ ]:
daily_sales = (
    orders_dates
    .resample("D", on="date")["amount"]
    .sum()
)

daily_sales

### Try it
Change `"D"` to `"W"` and compare the result.

## 11. Rolling summaries

Instead of creating another dataset, we can reuse the sales CSV. First calculate daily sales, then add a 3-day moving average.

In [ ]:
daily = (
    orders_dates
    .resample("D", on="date")["amount"]
    .sum()
    .reset_index(name="sales")
)

daily["rolling_3"] = (
    daily["sales"]
    .rolling(window=3)
    .mean()
)

daily.head(10).round(2)

The first two rolling values are missing because a complete 3-row window does not exist yet.

## 12. Create categories with `map()`, `replace()`, and `cut()`

Different tools are useful for different kinds of recoding:

- `map()` for dictionary-based labels
- `replace()` for selected substitutions
- `cut()` for numeric ranges

In [ ]:
orders_categories = orders.copy()

orders_categories["channel_label"] = orders_categories["channel"].map({
    "Web": "Online",
    "Email": "Online",
    "Store": "In person"
})

orders_categories["order_size"] = pd.cut(
    orders_categories["amount"],
    bins=[0, 60, 100, float("inf")],
    labels=["Small", "Medium", "Large"]
)

orders_categories[
    ["order_id", "channel", "channel_label", "amount", "order_size"]
]

In [ ]:
# A small replace() example
orders_categories["region_short"] = orders_categories["region"].replace({
    "North": "N",
    "South": "S",
    "West": "W"
})

orders_categories[["region", "region_short"]].drop_duplicates()

## 13. Safer joins with `merge()`

We can also practice joins using the CSV itself. Here we create a small customer lookup table from the unique customer IDs and segments, then merge it back to the order data.

Use `validate=` to check the expected relationship between keys, and `indicator=True` to make matching visible.

In [ ]:
orders_for_merge = orders[
    ["order_id", "customer_id", "amount"]
].copy()

# Ensure customer_id is strictly unique by using subset
customers = (
    orders[
        ["customer_id", "customer_segment"]
    ]
    .drop_duplicates(subset="customer_id")
)

merged = orders_for_merge.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

merged.head()

In [ ]:
# Check whether any order failed to match a customer.
merged["_merge"].value_counts()

## 14. Reusable workflows with `pipe()`

A small function can describe one transformation. `pipe()` lets that function fit naturally into a Pandas chain.

In [ ]:
def add_order_size(df):
    return df.assign(
        order_size=pd.cut(
            df["amount"],
            bins=[0, 60, 100, float("inf")],
            labels=["Small", "Medium", "Large"]
        )
    )


pipe_result = (
    orders
    .pipe(add_order_size)
    .sort_values("amount", ascending=False)
)

pipe_result[["order_id", "amount", "order_size"]]

## 15. Mini workflow: raw orders to report

Now combine several operations into one readable workflow.

In [ ]:
final_report = (
    orders
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .query("amount > 0")
    .assign(month=lambda df: df["date"].dt.to_period("M"))
    .groupby(["month", "region"], as_index=False)
    .agg(
        sales=("amount", "sum"),
        orders=("order_id", "count")
    )
    .sort_values(["month", "sales"], ascending=[True, False])
)

final_report

## 16. Practice challenge

Use the CSV-backed `orders` DataFrame and build a summary that does the following:

1. Convert `date` to datetime.
2. Keep orders with `amount >= 50`.
3. Create an `order_size` category:
   - Small: 0–60
   - Medium: 60–100
   - Large: above 100
4. Group by `region`.
5. Calculate:
   - `total_sales`
   - `average_order`
   - `orders`
6. Sort from highest to lowest `total_sales`.

Try to complete the workflow below using the real CSV data.

In [ ]:
# Write your practice solution here.

practice = orders.copy()

# Continue below:

## 17. Module 11 wrap-up

By the end of this notebook, you have used Pandas to:

- transform data with readable chains,
- summarize groups with `agg()` and `transform()`,
- reshape tables with pivot, melt, and explode,
- create time-based summaries,
- validate joins,
- and package repeated steps with `pipe()`.

**Next: Module 12 — Cleaning & EDA.** You will use these Pandas skills to investigate missing values, duplicates, inconsistent values, unusual observations, and patterns in real datasets.